# El Data Model de Python (Parte 2): clases que se comportan como colecciones

En el vídeo anterior aprendimos a integrar nuestras clases con Python implementando
*dunders* de **representación**, **igualdad** y **orden** (`__repr__`, `__eq__`, `__lt__`).

Ahora vamos a dar un paso más: hacer que una clase nuestra se comporte como una
**colección nativa**, de modo que podamos usar con ella `len()`, recorrerla con un
`for`, preguntar con `in` o acceder por índice con `[]`. Todo eso también son dunders.

La idea de fondo, que repetiremos varias veces, es: **no inventes una API paralela**.
Si Python ya tiene una forma estándar de "pedir la longitud" o "recorrer", aprovéchala.

El proyecto del vídeo será una clase **`Biblioteca`** que envuelve una lista de
`Libro` (la clase del vídeo anterior).

> 📚 Documentación oficial de referencia:
> **[Data Model — emular tipos contenedor](https://docs.python.org/3/reference/datamodel.html#emulating-container-types)**.

---

## 1. Punto de partida: `Libro` y una `Biblioteca` ingenua

Recuperamos una versión sencilla de `Libro` y creamos una `Biblioteca` que, de momento,
solo guarda la lista por dentro y expone métodos "a mano":

In [ ]:
class Libro:
    def __init__(self, titulo, anio):
        self.titulo = titulo
        self.anio = anio

    def __repr__(self):
        return f"Libro({self.titulo!r}, {self.anio})"


class Biblioteca:
    def __init__(self, libros=None):
        self.libros = list(libros) if libros else []

    # API "inventada": funciona, pero obliga a aprender nombres nuevos
    def cantidad(self):
        return len(self.libros)

    def obtener_libro(self, i):
        return self.libros[i]


bib = Biblioteca([Libro("1984", 1949), Libro("Dune", 1965)])
print(bib.cantidad())       # 2
print(bib.obtener_libro(0))  # Libro('1984', 1949)

Funciona, pero fíjate en el problema: quien use `Biblioteca` tiene que **aprender**
que la longitud se pide con `.cantidad()` y el acceso con `.obtener_libro(i)`. Son
nombres que nos hemos inventado. ¿Y si pudiéramos usar `len(bib)` y `bib[0]`, como con
una lista normal? Eso es justo lo que vamos a conseguir.

---

## 2. `__len__`: que funcione `len(biblioteca)`

Para que `len()` funcione sobre nuestro objeto, implementamos `__len__`:

In [ ]:
class Biblioteca:
    def __init__(self, libros=None):
        self.libros = list(libros) if libros else []

    def __len__(self):
        return len(self.libros)


bib = Biblioteca([Libro("1984", 1949), Libro("Dune", 1965)])
print(len(bib))  # 2  -> len() llama por debajo a __len__

---

## 3. `__iter__`: recorrer con `for libro in biblioteca`

Para poder iterar con un `for`, implementamos `__iter__`. En nuestro caso es muy
fácil: delegamos en el iterador de la lista interna con `iter(self.libros)`.

In [ ]:
class Biblioteca:
    def __init__(self, libros=None):
        self.libros = list(libros) if libros else []

    def __len__(self):
        return len(self.libros)

    def __iter__(self):
        return iter(self.libros)


bib = Biblioteca([Libro("1984", 1949), Libro("Dune", 1965)])
for libro in bib:        # for usa __iter__ por debajo
    print(libro)

> 💡 Al implementar `__iter__`, también funcionan "gratis" muchas cosas que usan
> iteración por debajo, como `list(bib)`, `sum(...)`, `max(...)` o las comprensiones
> (`[l.titulo for l in bib]`).

---

## 4. `__getitem__`: acceso por índice con `biblioteca[0]`

El dunder `__getitem__` es el que se ejecuta al usar los corchetes `[]`. Recibe el
índice (o la *slice*) y devolvemos el elemento correspondiente:

In [ ]:
class Biblioteca:
    def __init__(self, libros=None):
        self.libros = list(libros) if libros else []

    def __len__(self):
        return len(self.libros)

    def __iter__(self):
        return iter(self.libros)

    def __getitem__(self, i):
        return self.libros[i]


bib = Biblioteca([Libro("1984", 1949), Libro("Dune", 1965), Libro("Solaris", 1961)])
print(bib[0])      # primer libro
print(bib[-1])     # último libro
print(bib[0:2])    # ¡los slices también funcionan, porque la lista los soporta!

---

## 5. `__contains__`: preguntar con `libro in biblioteca`

Para que el operador `in` funcione, implementamos `__contains__`. Devuelve `True` o
`False` según si el elemento está dentro.

In [ ]:
class Libro:
    def __init__(self, titulo, anio):
        self.titulo = titulo
        self.anio = anio

    def __repr__(self):
        return f"Libro({self.titulo!r}, {self.anio})"

    def __eq__(self, otro):
        return (self.titulo, self.anio) == (otro.titulo, otro.anio)

    def __hash__(self):
        return hash((self.titulo, self.anio))


class Biblioteca:
    def __init__(self, libros=None):
        self.libros = list(libros) if libros else []

    def __contains__(self, libro):
        return libro in self.libros


bib = Biblioteca([Libro("1984", 1949), Libro("Dune", 1965)])
print(Libro("Dune", 1965) in bib)      # True
print(Libro("Solaris", 1961) in bib)   # False

> 💡 Aquí se ve por qué el `__eq__` del vídeo anterior era importante: `in` compara
> con `==`, así que `Libro("Dune", 1965) in bib` da `True` aunque no sea *el mismo*
> objeto que metimos, sino otro con los mismos datos.

---

## 6. `__add__`: combinar con `biblioteca1 + biblioteca2`

También podemos dar significado al operador `+` con `__add__`. Lo razonable es que
sumar dos bibliotecas devuelva una **nueva** biblioteca con los libros de ambas:

In [ ]:
class Biblioteca:
    def __init__(self, libros=None):
        self.libros = list(libros) if libros else []

    def __len__(self):
        return len(self.libros)

    def __add__(self, otra):
        return Biblioteca(self.libros + otra.libros)  # devuelve una NUEVA biblioteca


narrativa = Biblioteca([Libro("1984", 1949)])
ciencia_ficcion = Biblioteca([Libro("Dune", 1965), Libro("Solaris", 1961)])

completa = narrativa + ciencia_ficcion
print(len(completa))  # 3

---

## 7. Todo junto: una `Biblioteca` que "se siente de Python"

Reunimos todos los dunders en una sola clase y la usamos con la sintaxis nativa:

In [ ]:
class Biblioteca:
    def __init__(self, libros=None):
        self.libros = list(libros) if libros else []

    def __len__(self):
        return len(self.libros)

    def __iter__(self):
        return iter(self.libros)

    def __getitem__(self, i):
        return self.libros[i]

    def __contains__(self, libro):
        return libro in self.libros

    def __add__(self, otra):
        return Biblioteca(self.libros + otra.libros)


bib = Biblioteca([Libro("1984", 1949), Libro("Dune", 1965)])

print(len(bib))                      # 2
print(bib[0])                        # Libro('1984', 1949)
print(Libro("Dune", 1965) in bib)    # True
print([libro.titulo for libro in bib])  # ['1984', 'Dune']

---

## 8. La idea clave: no inventes APIs paralelas

Compara las dos formas de usar la biblioteca. La columna de la derecha no exige
aprender nada nuevo: es la sintaxis que **cualquier persona que sepa Python ya conoce**.

| Mal (API inventada) | Mejor (Data Model) |
|---------------------|--------------------|
| `biblioteca.cantidad()` | `len(biblioteca)` |
| `biblioteca.obtener_libro(0)` | `biblioteca[0]` |
| `biblioteca.contiene(libro)` | `libro in biblioteca` |
| `biblioteca.recorrer()` | `for libro in biblioteca` |

> 💡 Aprovechar la sintaxis nativa hace que **otras personas (y tú dentro de 6 meses)**
> usen tu clase sin tener que aprender una API nueva. Tu objeto se comporta como lo
> que parece, y eso es justo lo que esperamos de un buen diseño.

---

## 9. Resumen

En este vídeo hemos hecho que una clase nuestra se comporte como una **colección
nativa** implementando dunders del Data Model:

1. `__len__` → `len(obj)`.
2. `__iter__` → `for x in obj` (y gratis: `list()`, `sum()`, comprensiones...).
3. `__getitem__` → `obj[i]` (incluidos los *slices*).
4. `__contains__` → `x in obj` (se apoya en `__eq__`).
5. `__add__` → `obj1 + obj2`.

Y, sobre todo, una idea de diseño: **no inventes APIs paralelas**. Si Python ya tiene
una forma estándar de hacer algo, dale a tu clase los dunders para que la soporte.
Junto con el vídeo anterior, esto es lo que hace que una clase "se sienta de Python".

| Dunder | Sintaxis que habilita |
|--------|-----------------------|
| `__len__` | `len(obj)` |
| `__iter__` | `for x in obj` |
| `__getitem__` | `obj[i]`, `obj[a:b]` |
| `__contains__` | `x in obj` |
| `__add__` | `obj1 + obj2` |